# Domain 6: Governance, Risk, and Responsible Use
**Exam**: Claude Certified Associate – Foundations (CCAO-F)
**Weighting**: 15% of exam
**Depth**: deep

## Domain Overview

This domain covers recognizing appropriate vs. inappropriate use cases for Claude, applying
data sensitivity/regulatory/privacy considerations, following organizational AI policy, and
understanding the ethical implications of AI usage.

**Why it matters**: Associates are trusted to self-govern their own AI usage day-to-day; this domain
tests whether they can recognize risk before it becomes an incident.

**Key concepts to know**: appropriate use-case boundaries, data sensitivity handling, organizational
policy compliance, and AI ethics (bias, transparency, appropriate reliance).

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Identify appropriate and inappropriate use cases

Appropriate: drafting, summarizing, analyzing non-sensitive business content. Inappropriate:
using Claude as the sole basis for high-stakes decisions (hiring, medical, legal) without human
judgment, or processing data the organization has restricted.

In [ ]:
use_cases = [
    "Summarize a public blog post for a newsletter",
    "Make a final hiring decision based solely on Claude's assessment of a resume",
    "Draft talking points for a customer call",
    "Diagnose an employee's reported medical symptom",
]
for u in use_cases:
    print(u)

### Apply data sensitivity, regulatory, and privacy considerations

Regulated data (PII, health data, financial account numbers) generally should not be pasted
into a general-purpose AI tool without organizational approval and appropriate safeguards
(anonymization, approved enterprise deployment, data retention settings).

In [ ]:
def contains_likely_pii(text: str) -> bool:
    import re
    patterns = [r"\b\d{3}-\d{2}-\d{4}\b", r"\b\d{16}\b", r"[\w.]+@[\w.]+\.\w+"]
    return any(re.search(p, text) for p in patterns)

print(contains_likely_pii("Contact john.doe@example.com or SSN 123-45-6789"))
print(contains_likely_pii("Our Q3 revenue grew 12%"))

### Follow organizational AI policies and governance standards

Most organizations define what tools, data categories, and approval workflows apply to AI
usage. Associates are expected to know and follow their organization's specific policy rather than
assume general best practices are sufficient.

In [ ]:
policy = {
    "approved_tools": ["Claude Enterprise"],
    "restricted_data": ["customer PII", "unreleased financials", "health records"],
    "approval_required_for": ["external-facing content", "legal/compliance content"],
}
def is_compliant(tool: str, data_category: str) -> bool:
    return tool in policy["approved_tools"] and data_category not in policy["restricted_data"]

print(is_compliant("Claude Enterprise", "customer PII"))
print(is_compliant("Claude Enterprise", "marketing copy"))

### Understand the ethical implications of AI usage

Ethical considerations include bias (does the output unfairly favor one group?), transparency
(should people know AI was involved?), and appropriate reliance (not treating confident-sounding
output as automatically correct).

In [ ]:
ethical_checklist = [
    "Could this output disadvantage any group unfairly?",
    "Should the audience be told AI assisted in producing this?",
    "Am I treating this output with appropriate skepticism given the stakes?",
]
for q in ethical_checklist:
    print("-", q)

## Mini Hands-On Project

**Covers**: all Domain 6 sub-objectives.

**Scenario**: A colleague asks you to paste a spreadsheet of customer names and support ticket
details into Claude to find common complaint themes. Walk through the governance decision.

### Step 1: Screen for sensitive data

In [ ]:
sample_row = "John Smith, john.smith@email.com, complained about billing on 2025-03-01"
print(contains_likely_pii(sample_row))

### Step 2: Apply policy and decide on redaction

In [ ]:
redacted_row = "Customer A, [redacted email], complained about billing on 2025-03-01"
print(contains_likely_pii(redacted_row))
print(is_compliant("Claude Enterprise", "customer PII" if contains_likely_pii(sample_row) else "marketing copy"))

### Step 3: Proceed with the analysis on redacted data

In [ ]:
theme_analysis = call_claude(f"Identify common complaint themes in this ticket data: {redacted_row}")
print(theme_analysis)

## Lab Exercises

### Exercise 1: Use-case classifier

**Objective**: Classify 6 use cases as appropriate/inappropriate with reasoning.

**Produce**: A list of 6 classifications with one-sentence justifications.

In [ ]:
use_cases = [
    "Draft an internal FAQ from public documentation",
    "Auto-approve a loan application based only on Claude's recommendation",
    "Summarize anonymized survey feedback",
    "Generate a performance review with no manager input",
    "Brainstorm marketing taglines",
    "Screen resumes to eliminate candidates with no human review",
]
# TODO: classify each as appropriate/inappropriate with a one-sentence reason
for u in use_cases:
    classification = "..."  # TODO
    reason = "..."  # TODO
    print(u, "->", classification, "--", reason)

# Expected output: High-stakes, human-judgment-required cases (loans, performance reviews, resume screening without review) marked inappropriate.

In [ ]:
# STRETCH GOAL: for each "inappropriate" case, propose a modified version that WOULD be appropriate
# (e.g., adding required human review).
for u in use_cases:
    pass  # TODO: if inappropriate, propose a safe alternative

# Expected output: A safe, human-in-the-loop alternative proposed for each inappropriate use case.

### Exercise 2: PII scanner improvement

**Objective**: Extend `contains_likely_pii` to detect one more sensitive pattern.

**Produce**: An updated function that also flags phone numbers, tested against 3 examples.

In [ ]:
import re

def contains_likely_pii_v2(text: str) -> bool:
    patterns = [r"\b\d{3}-\d{2}-\d{4}\b", r"\b\d{16}\b", r"[\w.]+@[\w.]+\.\w+"]
    # TODO: add a regex pattern for US phone numbers like 555-123-4567 or (555) 123-4567
    return any(re.search(p, text) for p in patterns)

print(contains_likely_pii_v2("Call me at 555-123-4567"))
print(contains_likely_pii_v2("No sensitive info here"))

# Expected output: Correctly flags the phone-number example as containing PII, and the clean example as not.

In [ ]:
# STRETCH GOAL: write a redaction function that replaces any matched PII with "[redacted]".
def redact_pii(text: str) -> str:
    # TODO: use re.sub with the same patterns to replace matches with "[redacted]"
    pass
print(redact_pii("Call me at 555-123-4567 or email john@example.com"))

# Expected output: A version of the input string with phone numbers and emails replaced by [redacted].

### Exercise 3: Policy compliance checker

**Objective**: Extend the policy dict and compliance function to cover a new tool.

**Produce**: An updated compliance check covering a second approved tool with different restrictions.

In [ ]:
policy_v2 = {
    "Claude Enterprise": {"restricted_data": ["customer PII", "unreleased financials", "health records"]},
    "Claude Free (personal account)": {"restricted_data": ["customer PII", "unreleased financials", "health records", "internal strategy docs"]},
}
# TODO: write is_compliant_v2(tool, data_category) using policy_v2
def is_compliant_v2(tool: str, data_category: str) -> bool:
    pass

print(is_compliant_v2("Claude Free (personal account)", "internal strategy docs"))
print(is_compliant_v2("Claude Enterprise", "internal strategy docs"))

# Expected output: Personal accounts correctly restricted from more categories than the enterprise tool.

In [ ]:
# STRETCH GOAL: write a function that recommends the correct approved tool given a data category.
def recommend_tool(data_category: str, policy: dict) -> str:
    # TODO: return the first tool in policy for which data_category is NOT restricted
    pass
print(recommend_tool("internal strategy docs", policy_v2))

# Expected output: Recommends 'Claude Enterprise' since it's the only one that allows internal strategy docs.

### Exercise 4: Bias spot-check

**Objective**: Generate and review output for potential bias in a hiring-adjacent scenario.

**Produce**: A generated output plus a written bias review.

In [ ]:
prompt = "Write a short candidate profile summary for a software engineering applicant based on this resume text: '10 years experience, worked at 3 startups, has a gap year for family care.'"
output = call_claude(prompt)
print(output)

# TODO: write a short review checking whether the summary penalizes the gap year unfairly, or introduces
# any other unjustified assumptions
bias_review = "..."
print(bias_review)

# Expected output: A thoughtful review identifying whether the gap year (or any other detail) was treated unfairly or neutrally.

In [ ]:
# STRETCH GOAL: ask Claude to review its own output for potential bias, and compare to your manual review.
self_review = call_claude(f"Review this candidate summary for any unfair assumptions or bias: {output}")
print(self_review)

# Expected output: A self-review from Claude that can be compared against the manual review for agreement/disagreement.

### Exercise 5: Governance decision memo

**Objective**: Write a short memo deciding whether a new use case can proceed, and under what conditions.

**Produce**: A memo with a go/no-go decision and any required safeguards.

In [ ]:
proposed_use = "Use Claude to draft responses to employee HR complaints, to be sent without review to save time."

# TODO: decide go/no-go and required safeguards (e.g., mandatory human review, redaction, approval)
decision = "..."  # TODO
safeguards = []  # TODO

print(decision)
print(safeguards)

# Expected output: A 'no-go without safeguards' decision, with mandatory human review as a required safeguard given the sensitivity.

In [ ]:
# STRETCH GOAL: draft the actual policy language for this use case's approval workflow.
policy_language = call_claude(f"Draft a short policy paragraph describing the required approval workflow for: {proposed_use}")
print(policy_language)

# Expected output: A clear, specific policy paragraph describing the required review/approval workflow.

## Self-Assessment: Governance, Risk, and Responsible Use

Answer each question, then check the Answers cell below.


**Q1.** What distinguishes an appropriate Claude use case from an inappropriate one, in terms of decision stakes?

**Q2.** What should you do before pasting a spreadsheet containing customer names and emails into a general Claude conversation?

**Q3.** Why is it important to follow your organization's specific AI governance policy rather than general best practices alone?

**Q4.** What is meant by 'appropriate reliance' on AI output as an ethical consideration?

**Q5.** True or False: as long as the output sounds confident, it's safe to treat it as accurate for high-stakes decisions.

## Answers


**A1.** Appropriate use cases keep a human in the loop for high-stakes or judgment-heavy decisions; inappropriate ones let AI output stand alone for such decisions.

**A2.** Screen for and redact/anonymize sensitive personal data (PII) consistent with organizational policy before uploading it.

**A3.** Because organizational policy reflects specific legal, contractual, and risk considerations that general best practices may not cover.

**A4.** Treating AI output with a level of skepticism and verification proportional to the stakes, rather than trusting it automatically.

**A5.** False — confidence in tone is not a reliable signal of accuracy, especially for high-stakes decisions.

---
**Next**: [Domain 7: Troubleshooting and Optimization](./07-troubleshooting-and-optimization.ipynb)
**Back**: [README](./README.md)